# 工具的使用案例

### 案例1 使用args_schema

举例:

In [7]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_core.tools import tool
from rich import print as rprint
from config.config import DeepSeekModel
from pydantic import BaseModel,Field

model = init_chat_model(
    base_url=DeepSeekModel.deepseek_base_url,
    api_key=DeepSeekModel.deepseek_api_key,
    model=DeepSeekModel.deepseek_model,
    model_provider="openai"
)



In [8]:
# 定义工具模型
class GetWeatherSchema(BaseModel):
    city:str = Field(
        min_length=2,
        max_length=10,
        default="沈阳",
        description="城市名字"
    )
    if_forecast:bool = Field(
        default=True,
        description="是否包含明日天气"
    )

# 定义工具
@tool(
    description="获取具体城市天气",
    args_schema=GetWeatherSchema
)
def get_weather(city:str,if_forecast:bool) -> str:

    res = f"{city}今天天气不错\n"
    if if_forecast:
        res += f"明天{city}下雨"

    return res



In [17]:
# 调用工具

# 绑定工具到模型
model_with_tools = model.bind_tools([get_weather])

# 定义消息列表
messages:list[BaseMessage] = [
    HumanMessage("今天天气怎么样？明天呢？")
]

response = model_with_tools.invoke(messages)

# 将返回的aimmessage添加到消息列表
messages.append(response)

# 遍历得到工具参数列表
if not response.tool_calls:
    rprint("无需调用工具")
    rprint(response.content)

for tool in response.tool_calls:
    if tool["name"] == "get_weather":
        # 主动调用工具(因为大模型不会主动调用工具，只有agent才可以)
        tool_message = get_weather.invoke(tool)
        # 将工具返回的toolmessage添加到消息列表
        messages.append(tool_message)

# 将全部历史消息发送给ai
final_response = model_with_tools.invoke(messages)

# 整合全部消息
messages.append(final_response)

# 遍历得到全部消息
for msg in messages:
    msg.pretty_print()
    rprint(msg.content)

messages.clear()

================================ Human Message =================================

今天天气怎么样？明天呢？


今天天气怎么样？明天呢？

================================== Ai Message ==================================

我来帮您查询天气情况。
Tool Calls:
  get_weather (call_00_lWS58OMkltWdmFkfW2gR3470)
 Call ID: call_00_lWS58OMkltWdmFkfW2gR3470
  Args:
    city: 沈阳
    if_forecast: True


我来帮您查询天气情况。

================================= Tool Message =================================
Name: get_weather

沈阳今天天气不错
明天沈阳下雨


沈阳今天天气不错
明天沈阳下雨

================================== Ai Message ==================================

为您查询到**沈阳**的天气情况：

- **今天**：天气不错 ☀️，适合外出活动。
- **明天**：有雨 🌧️，出门记得带伞，注意添衣。

由于您没有指定城市，我默认查询了沈阳。如果您想了解其他城市的天气，告诉我城市名称即可～


为您查询到**沈阳**的天气情况：

- **今天**：天气不错 ☀️，适合外出活动。
- **明天**：有雨 🌧️，出门记得带伞，注意添衣。

由于您没有指定城市，我默认查询了沈阳。如果您想了解其他城市的天气，告诉我城市名称即可～